In [ ]:
# ==============================================================================
# STEP 1: Dependencies & Environment Setup
# ==============================================================================
!pip install -q --upgrade "datasets==3.6.0" soundfile diffusers transformers accelerate

import torch
from transformers import pipeline
from diffusers import DiffusionPipeline
from google.colab import userdata
from huggingface_hub import login

# Authenticate Hugging Face Hub access
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=False)
    print("Hugging Face Login Successful!")
except Exception as e:
    print("Ensure HF_TOKEN is activated in the Colab Secrets key tab.")

# ==============================================================================
# STEP 2: Text Pipelines (NLP Inference)
# ==============================================================================

# 1. Sentiment Analysis
print("\n--- Running Sentiment Analysis ---")
sentiment_pipe = pipeline("sentiment-analysis", device=0)
res = sentiment_pipe("I am super excited to be on the way to LLM mastery!")
print(f"Result: {res}")

# 2. Named Entity Recognition (NER)
print("\n--- Running Named Entity Recognition ---")
ner_pipe = pipeline("ner", device=0)
entities = ner_pipe("AI engineers are learning Hugging Face pipelines in Google Colab from Donna.")
for entity in entities:
    print(f"Token: {entity['word']} | Entity: {entity['entity']} | Score: {entity['score']:.4f}")

# 3. Question Answering (Option A: Modern Instruction-Tuned Model)
print("\n--- Running Question Answering ---")
qa_pipe = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device=0)

messages = [
    {"role": "system", "content": "Answer the question concisely based on the context provided."},
    {"role": "user", "content": "Context: Pipelines are a high-level API for inference of LLMs with common tasks.\n\nQuestion: What are Hugging Face pipelines?"}
]

qa_result = qa_pipe(messages, max_new_tokens=40)
print(f"Answer: {qa_result[0]['generated_text'][-1]['content']}")

# 4. Zero-Shot Classification
print("\n--- Running Zero-Shot Classification ---")
classifier = pipeline("zero-shot-classification", device=0)
text_to_classify = "Hugging Face Transformers library is amazing!"
candidate_labels = ["technology", "sports", "politics"]
cls_result = classifier(text_to_classify, candidate_labels)
print(f"Labels: {cls_result['labels']}")
print(f"Scores: {cls_result['scores']}")

# 5. Text Generation (GPT-2 Demo)
print("\n--- Running Text Generation ---")
text_gen = pipeline("text-generation", model="gpt2", device=0)
gen_output = text_gen("If there is one thing I want you to remember about using Hugging Face pipelines, it's", max_new_tokens=40)
print(f"Generated: {gen_output[0]['generated_text']}")

# ==============================================================================
# STEP 3: Multimodal Pipelines (Audio & Image)
# ==============================================================================

# 6. Text-to-Speech (Audio)
print("\n--- Running Text-to-Speech Pipeline ---")
import soundfile as sf

tts_pipe = pipeline("text-to-speech", model="microsoft/speecht5_tts", device=0)
speaker_embedding = torch.zeros((1, 512)).to(0)
speech = tts_pipe("Hi to an artificial intelligence engineer on the way to mastery.", forward_params={"speaker_embeddings": speaker_embedding})
sf.write("day2_audio.wav", speech["audio"], speech["sampling_rate"])
print("Saved audio to day2_audio.wav")

# 7. Image Generation (Stable Diffusion XL Turbo)
print("\n--- Running Image Diffusion Pipeline ---")
diff_pipe = DiffusionPipeline.from_pretrained(
    "stabilityai/sdxl-turbo",
    torch_dtype=torch.float16,
    variant="fp16"
).to("cuda")

image = diff_pipe(prompt="A class of students learning AI engineering in a vibrant pop art style", num_inference_steps=1, guidance_scale=0.0).images[0]
image.save("day2_popart.png")
print("Saved rendered image to day2_popart.png")